# Dataset Collection & Wrangling

## Table of Contents
* [Requirements](#requirements)
* [Data Collection & Time Alignment](#data-collection)
    * [Weather Data](#weather-data)
    * [Day-Ahead Price Data](#day-ahead-price-data)
    * [Production Data](#production-data)
    * [Cross-Border Electricity Trading Data](#cross-border-electricity-trading-data)
* [Data Concatenation](#data-concatenation)
* [Covariate Lagging & Train-Test Split](#covariate-lagging-&-train-test-split)
* [Data Dictionary](#data-dictionary)

----

## Requirements

In [1]:
import sys
from pathlib import Path
import requests
from joblib import dump
from functools import reduce

import pandas as pd

from sklearn.preprocessing import StandardScaler

project_root = Path().resolve().parents[0]
sys.path.append(str(project_root))
from src.data_setup import save_splits

pd.set_option('display.max_rows', 100) 
pd.set_option('display.max_columns', 100) 

/Users/benleidig/Downloads/public_repos/de_lu_epf/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


----

## Data Collection

### Weather Data

In [2]:
coordinates = [
    (52.5200, 13.4050),  # Berlin
    (53.5488, 9.9872),   # Hamburg
    (48.1351, 11.5820),  # Munich
    (50.9375, 6.9603)    # Cologne
]
weather_variables = [
    'precipitation',
    'cloud_cover',
    'sunshine',
    'temperature',
    'relative_humidity'
]

In [3]:
%%time

url = 'https://api.brightsky.dev/weather'
parameters = {
    'date':'2018-01-01T00:00:00Z',
    'last_date':'2025-01-01T00:00:00Z'
}
concat_list = []
for lat, lon in coordinates:
    parameters['lat'] = lat
    parameters['lon'] = lon
    data = requests.get(url, parameters).json()
    temp = pd.DataFrame(data['weather'])[['timestamp'] + weather_variables]
    concat_list.append(temp)
weather = pd.concat(concat_list)
weather['datetime'] = pd.to_datetime(weather['timestamp'], format='ISO8601', utc=True) + pd.Timedelta(hours=-1)
weather = weather\
    .drop(columns='timestamp')\
    .groupby('datetime', as_index=False)\
    .agg('mean')
print(weather.shape)
weather.head()

(61369, 6)
CPU times: user 3.07 s, sys: 214 ms, total: 3.29 s
Wall time: 5min 56s


,datetime,precipitation,cloud_cover,sunshine,temperature,relative_humidity
0,2017-12-31 23:00:00+00:00,0.450,100.0,NaN,8.275,82.00
1,2018-01-01 00:00:00+00:00,0.225,81.0,NaN,8.950,76.25
2,2018-01-01 01:00:00+00:00,0.100,75.0,NaN,8.450,76.25
3,2018-01-01 02:00:00+00:00,0.000,90.5,0.0,8.875,72.50
4,2018-01-01 03:00:00+00:00,0.000,93.5,0.0,8.875,74.00


### Day-Ahead Price Data

In [4]:
%%time

url = 'https://api.energy-charts.info/price'
parameters = {
    'start':'2018-01-01T00:00:00Z',
    'end':'2024-12-31T23:00:00Z'
}
data = requests.get(url, params=parameters).json()
data.pop('license_info'); data.pop('unit'); data.pop('deprecated')
price = pd.DataFrame(data)
price['datetime'] = pd.to_datetime(price['unix_seconds'], unit='s', utc=True)
price = price.drop(columns='unix_seconds')
print(price.shape)
price.head()

(61368, 2)
CPU times: user 120 ms, sys: 23.8 ms, total: 144 ms
Wall time: 7.04 s


,price,datetime
0,NaN,2018-01-01 00:00:00+00:00
1,NaN,2018-01-01 01:00:00+00:00
2,NaN,2018-01-01 02:00:00+00:00
3,NaN,2018-01-01 03:00:00+00:00
4,NaN,2018-01-01 04:00:00+00:00


### Production Data

In [5]:
%%time

url = 'https://api.energy-charts.info/public_power'
parameters = {
    'start':'2018-01-01T00:00:00Z',
    'end':'2025-01-01T00:00:00Z'
}
data = requests.get(url, params=parameters).json()
public_power = pd.DataFrame()
public_power['datetime'] = pd.to_datetime(data['unix_seconds'], unit='s', utc=True) + pd.Timedelta(hours=-1)
for production_type_data in data['production_types']:
    col = production_type_data['name'].lower().replace(' ', '_').replace('-', '_')
    public_power[col] = production_type_data['data']
public_power = public_power[
    public_power['datetime'].dt.minute == 0
].drop(columns=['hydro_pumped_storage_consumption', 'cross_border_electricity_trading'])
print(public_power.shape)
public_power.head()

(61369, 21)
CPU times: user 1.94 s, sys: 82.4 ms, total: 2.03 s
Wall time: 18.5 s


,datetime,nuclear,hydro_run_of_river,biomass,fossil_brown_coal_/_lignite,fossil_hard_coal,fossil_oil,fossil_coal_derived_gas,fossil_gas,geothermal,hydro_water_reservoir,hydro_pumped_storage,others,waste,wind_offshore,wind_onshore,solar,load,residual_load,renewable_share_of_load,renewable_share_of_generation
0,2017-12-31 23:00:00+00:00,4943.3,1904.2,4658.9,7009.5,1692.3,187.8,504.7,2492.8,19.9,43.8,757.3,435.1,1219.7,2912.2,29234.7,0.0,44615.7,12468.7,88.1,67.8
4,2018-01-01 00:00:00+00:00,4580.7,1896.4,4625.7,6989.8,1755.4,187.5,538.8,2481.9,20.0,44.9,707.9,433.2,1102.6,3106.4,30054.1,0.0,43314.8,10154.3,92.9,68.8
8,2018-01-01 01:00:00+00:00,4907.7,1886.1,4627.1,6936.8,1780.7,187.1,494.1,2512.3,19.8,42.5,77.1,427.9,1110.4,3101.8,30603.3,0.0,42648.9,8943.7,95.6,69.5
12,2018-01-01 02:00:00+00:00,4747.9,1877.4,4634.2,6863.4,1753.9,187.2,463.6,2503.2,19.0,44.2,55.8,428.1,1135.6,3233.5,30848.4,0.0,42424.4,8342.6,97.0,70.0
16,2018-01-01 03:00:00+00:00,4751.0,1867.6,4627.2,6892.9,1766.4,187.1,496.3,2509.9,18.8,45.1,62.1,428.6,1190.1,3301.0,31228.4,0.4,42519.0,7989.2,97.9,70.1


### Cross-Border Electricity Trading Data

In [6]:
%%time

url = 'https://api.energy-charts.info/cbet'
parameters = {
    'start':'2018-01-01T00:00:00Z',
    'end':'2025-01-01T00:00:00Z'
}
data = requests.get(url, parameters).json()
cbet = pd.DataFrame()
cbet['datetime'] = pd.to_datetime(data['unix_seconds'], unit='s', utc=True) + pd.Timedelta(hours=-1)
for country_data in data['countries']:
    col = country_data['name'].lower().replace(' ', '_').replace('-', '_') + '_cbet'
    cbet[col] = country_data['data']
cbet = cbet[
    cbet['datetime'].dt.minute == 0
]
print(cbet.shape)
cbet.head()

(61369, 13)
CPU times: user 1.15 s, sys: 37.2 ms, total: 1.19 s
Wall time: 9.97 s


,datetime,austria_cbet,belgium_cbet,czech_republic_cbet,denmark_cbet,france_cbet,luxembourg_cbet,netherlands_cbet,norway_cbet,poland_cbet,sweden_cbet,switzerland_cbet,sum_cbet
0,2017-12-31 23:00:00+00:00,-4.594,NaN,-0.981,-2.1,-6.597,-0.291,-0.406,NaN,0.0,-0.076,-0.800,-15.844
4,2018-01-01 00:00:00+00:00,-4.619,NaN,-1.409,-2.1,-6.803,-0.270,-0.200,NaN,0.0,-0.076,-0.800,-16.276
8,2018-01-01 01:00:00+00:00,-4.502,NaN,-1.825,-2.1,-6.166,-0.256,-0.714,NaN,0.0,-0.076,-0.723,-16.362
12,2018-01-01 02:00:00+00:00,-4.600,NaN,-1.785,-2.1,-5.913,-0.250,-0.118,NaN,0.0,-0.076,-0.800,-15.642
16,2018-01-01 03:00:00+00:00,-4.618,NaN,-1.811,-2.1,-5.738,-0.252,-0.201,NaN,0.0,-0.076,-0.790,-15.586


In [21]:
cbet[cbet['datetime'].dt.year >= 2019]

,datetime,austria_cbet,belgium_cbet,czech_republic_cbet,denmark_cbet,france_cbet,luxembourg_cbet,netherlands_cbet,norway_cbet,poland_cbet,sweden_cbet,switzerland_cbet,sum_cbet
35044,2019-01-01 00:00:00+00:00,-4.264,NaN,-0.710,-0.183,-3.642,-0.360,-1.176,NaN,0.000,-0.076,-0.8,-11.211
35048,2019-01-01 01:00:00+00:00,-4.270,NaN,-1.100,-0.307,-3.083,-0.350,-1.428,NaN,0.000,-0.067,-0.8,-11.405
35052,2019-01-01 02:00:00+00:00,-4.961,NaN,-1.137,-0.412,-3.130,-0.344,-1.492,NaN,0.000,-0.067,-0.8,-12.343
35056,2019-01-01 03:00:00+00:00,-4.984,NaN,-1.100,-0.676,-4.060,-0.360,-0.862,NaN,0.000,-0.067,-0.8,-12.910
35060,2019-01-01 04:00:00+00:00,-4.817,NaN,-1.000,-1.441,-4.641,-0.336,-0.844,NaN,0.000,-0.067,-0.8,-13.946
...,...,...,...,...,...,...,...,...,...,...,...,...,...
245456,2024-12-31 19:00:00+00:00,-2.901,-1.000,-0.762,0.677,-3.427,-0.342,-0.419,0.659,-0.250,0.000,-0.8,-8.566
245460,2024-12-31 20:00:00+00:00,-3.088,-1.000,-0.856,0.806,-2.900,-0.348,-0.361,0.070,-0.222,0.081,-0.8,-8.618
245464,2024-12-31 21:00:00+00:00,-3.573,-1.000,-1.012,0.483,-2.102,-0.315,-0.098,-0.519,-0.320,-0.507,-0.8,-9.762
245468,2024-12-31 22:00:00+00:00,-3.040,-1.000,-0.828,-0.302,-3.351,-0.297,0.172,-1.119,-0.094,-0.507,-0.8,-11.166


----

## Data Concatenation

In [ ]:
df = reduce(
    lambda l, r : l.merge(r, on='datetime', how='outer'),
    [weather, price, public_power, cbet]
).sort_values(by='datetime', ascending=True)

df['rolling_mean'] = df['price'].rolling(window=24*7*4).mean()
df['rolling_var'] = df['price'].rolling(window=24*7*4).var()
df = df.set_index('datetime')

df.to_csv('../data/raw/dataset.csv', index=False)
print(df.shape)
df.head()

(61369, 40)


,precipitation,cloud_cover,sunshine,temperature,relative_humidity,price,nuclear,hydro_run_of_river,biomass,fossil_brown_coal_/_lignite,fossil_hard_coal,fossil_oil,fossil_coal_derived_gas,fossil_gas,geothermal,hydro_water_reservoir,hydro_pumped_storage,others,waste,wind_offshore,wind_onshore,solar,load,residual_load,renewable_share_of_load,renewable_share_of_generation,austria_cbet,belgium_cbet,czech_republic_cbet,denmark_cbet,france_cbet,luxembourg_cbet,netherlands_cbet,norway_cbet,poland_cbet,sweden_cbet,switzerland_cbet,sum_cbet,rolling_mean,rolling_var
datetime,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2017-12-31 23:00:00+00:00,0.450,100.0,NaN,8.275,82.00,NaN,4943.3,1904.2,4658.9,7009.5,1692.3,187.8,504.7,2492.8,19.9,43.8,757.3,435.1,1219.7,2912.2,29234.7,0.0,44615.7,12468.7,88.1,67.8,-4.594,NaN,-0.981,-2.1,-6.597,-0.291,-0.406,NaN,0.0,-0.076,-0.800,-15.844,NaN,NaN
2018-01-01 00:00:00+00:00,0.225,81.0,NaN,8.950,76.25,NaN,4580.7,1896.4,4625.7,6989.8,1755.4,187.5,538.8,2481.9,20.0,44.9,707.9,433.2,1102.6,3106.4,30054.1,0.0,43314.8,10154.3,92.9,68.8,-4.619,NaN,-1.409,-2.1,-6.803,-0.270,-0.200,NaN,0.0,-0.076,-0.800,-16.276,NaN,NaN
2018-01-01 01:00:00+00:00,0.100,75.0,NaN,8.450,76.25,NaN,4907.7,1886.1,4627.1,6936.8,1780.7,187.1,494.1,2512.3,19.8,42.5,77.1,427.9,1110.4,3101.8,30603.3,0.0,42648.9,8943.7,95.6,69.5,-4.502,NaN,-1.825,-2.1,-6.166,-0.256,-0.714,NaN,0.0,-0.076,-0.723,-16.362,NaN,NaN
2018-01-01 02:00:00+00:00,0.000,90.5,0.0,8.875,72.50,NaN,4747.9,1877.4,4634.2,6863.4,1753.9,187.2,463.6,2503.2,19.0,44.2,55.8,428.1,1135.6,3233.5,30848.4,0.0,42424.4,8342.6,97.0,70.0,-4.600,NaN,-1.785,-2.1,-5.913,-0.250,-0.118,NaN,0.0,-0.076,-0.800,-15.642,NaN,NaN
2018-01-01 03:00:00+00:00,0.000,93.5,0.0,8.875,74.00,NaN,4751.0,1867.6,4627.2,6892.9,1766.4,187.1,496.3,2509.9,18.8,45.1,62.1,428.6,1190.1,3301.0,31228.4,0.4,42519.0,7989.2,97.9,70.1,-4.618,NaN,-1.811,-2.1,-5.738,-0.252,-0.201,NaN,0.0,-0.076,-0.790,-15.586,NaN,NaN


In [23]:
print(df.dtypes)
print(f'Total NaNs: {df.isna().sum().sum()}\n')
print(df.isna().sum())

precipitation                    float64
cloud_cover                      float64
sunshine                         float64
temperature                      float64
relative_humidity                float64
price                            float64
nuclear                          float64
hydro_run_of_river               float64
biomass                          float64
fossil_brown_coal_/_lignite      float64
fossil_hard_coal                 float64
fossil_oil                       float64
fossil_coal_derived_gas          float64
fossil_gas                       float64
geothermal                       float64
hydro_water_reservoir            float64
hydro_pumped_storage             float64
others                           float64
waste                            float64
wind_offshore                    float64
wind_onshore                     float64
solar                            float64
load                             float64
residual_load                    float64
renewable_share_

----

## Covariate Lagging & Train-Test Split

In [9]:
# df = pd.read_csv('../data/raw/dataset.csv')
# df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
# df = df.set_index('datetime')

In [28]:
## selecting features based on domain knowledge
subset = [
    'price',
    'rolling_mean', 'rolling_var',
    'precipitation', 'temperature', 'relative_humidity',
    'load', 'renewable_share_of_load', 'renewable_share_of_generation',
    'sum_cbet'
]
df_subset = df[subset]

## lagging features to abide by the "day-ahead" constraint
for col in subset[3:]:
    df_subset.loc[:, col] = df_subset[col].shift(12)

df_subset = df_subset[df_subset.index.year >= 2019]
df_subset.to_pickle('../data/processed/processed_dataset.pkl')
print(df_subset.shape)
df_subset.head()

(52608, 10)


,price,rolling_mean,rolling_var,precipitation,temperature,relative_humidity,load,renewable_share_of_load,renewable_share_of_generation,sum_cbet
datetime,,,,,,,,,,
2019-01-01 00:00:00+00:00,10.07,48.899777,431.477251,0.250,7.05,92.50,52750.6,30.6,31.7,-0.605
2019-01-01 01:00:00+00:00,-4.08,48.847485,435.183304,0.100,7.25,91.25,51944.2,31.7,32.2,-1.946
2019-01-01 02:00:00+00:00,-9.91,48.791116,439.675759,0.075,7.25,92.25,53075.2,33.1,33.3,-1.017
2019-01-01 03:00:00+00:00,-7.41,48.732649,443.953378,0.050,7.05,94.00,56323.7,34.1,34.5,-1.964
2019-01-01 04:00:00+00:00,-12.55,48.654643,449.427190,0.000,7.15,93.50,56937.8,38.0,37.8,-3.168


In [25]:
## 2019 - 2023 is train_val : 2024 is test
## saving train and test splits (unstandardized, standardized, scalers)
save_splits(
    df=df_subset,
    time_splits=[
        (   # train_start            # train_end
            '2019-01-01 00:00:00+00:00', '2022-12-31 23:00:00+00:00',
            '2023-01-01 00:00:00+00:00', '2023-12-31 23:00:00+00:00'
        )   # val_start                 # val_end
    ],
    f_names=[
        ('df_train', 'df_val')
    ]
)
save_splits(
    df=df_subset,
    time_splits=[
        (   # train_val_start            # train_val_end
            '2019-01-01 00:00:00+00:00', '2023-12-31 23:00:00+00:00',
            '2024-01-01 00:00:00+00:00', '2024-12-31 23:00:00+00:00'
        )   # test_start                 # test_end
    ],
    f_names=[
        ('df_train_val', 'df_test')
    ]
)
save_splits(
    df=df_subset,
    time_splits=[
        (   # train_start            # train_end
            '2019-01-01 00:00:00+00:00', '2022-12-31 23:00:00+00:00',
            '2024-01-01 00:00:00+00:00', '2024-12-31 23:00:00+00:00'
        )   # test_start                 # test_end
    ],
    f_names=[
        ('df_train', 'df_test')
    ]
)

## saving train and val splits (unstandardized, standardized, scalers)
save_splits(
    df=df_subset,
    time_splits=[
        # split 1
        (   # train_start                # train_end
            '2019-01-01 00:00:00+00:00', '2022-12-31 23:00:00+00:00',
            '2023-01-01 00:00:00+00:00', '2023-06-30 23:00:00+00:00'
        ),  # val_start                  # val_end
        # split 2
        (   # train_start                # train_end
            '2019-01-01 00:00:00+00:00', '2023-06-30 23:00:00+00:00',
            '2023-07-01 00:00:00+00:00', '2023-12-31 23:00:00+00:00'
        ),  # val_start                  # val_end
    ],
    f_names=[
        ('df_train1', 'df_val1'),   # split 1
        ('df_train2', 'df_val2')    # split 2
    ]
)

----

## Data Dictionary

In [ ]:
descriptions_units = {
    'datetime':                         ('ISO 8601-formatted timestamp of this record in UTC.', 'YYYY-MM-DDThh:mm:ss'),
    'precipitation':                    ('Total precipitation during the following hour.', 'mm'),
    'cloud_cover':                      ('Total cloud cover an hour after the timestamp.', '%'),
    'sunshine':                         ('Sunshine duration during the following hour.', 'min'),
    'temperature':                      ('Air temperature an hour after the timestamp, 2 m above the ground.', 'ºC'),
    'relative_humidity':                ('Relative humidity an hour after the timestamp.', '%'),
    'price':                            ('The day-ahead spot market price for the DE-LU bidding zone. Takes the last 15-minute interval value and assigns to the start of the hour interval.', 'EUR/MWh'),
    'hydro_run_of_river':               ('Electricity generation from hydro run-of-river recorded at the end of the following hour.', 'MW'),
    'biomass':                          ('Electricity generation from biomass recorded at the end of the following hour.', 'MW'),
    'fossil_brown_coal_/_lignite':      ('Electricity generation from fossil brown coal / lignite recorded at the end of the following hour.', 'MW'),
    'fossil_hard_coal':                 ('Electricity generation from fossil hard coal recorded at the end of the following hour.', 'MW'),
    'fossil_oil':                       ('Electricity generation from fossil oil recorded at the end of the following hour.', 'MW'),
    'fossil_coal_derived_gas':          ('Electricity generation from fossil coal-derived gas recorded at the end of the following hour.', 'MW'),
    'fossil_gas':                       ('Electricity generation from fossil gas recorded at the end of the following hour.', 'MW'),
    'geothermal':                       ('Electricity generation from geothermal recorded at the end of the following hour.', 'MW'),
    'hydro_water_reservoir':            ('Electricity generation from hydro water reservoir recorded at the end of the following hour.', 'MW'),
    'hydro_pumped_storage':             ('Electricity generation from pumped storage recorded at the end of the following hour.', 'MW'),
    'others':                           ('Electricity generation from other sources recorded at the end of the following hour.', 'MW'),
    'waste':                            ('Electricity generation from waste recorded at the end of the following hour.', 'MW'),
    'wind_offshore':                    ('Electricity generation from wind offshore recorded at the end of the following hour.', 'MW'),
    'wind_onshore':                     ('Electricity generation from wind onshore recorded at the end of the following hour.', 'MW'),
    'solar':                            ('Electricity generation from solar recorded at the end of the following hour.', 'MW'),
    'load':                             ('Instantaneous grid load recorded at the end of the following hour.', 'MW'),
    'residual_load':                    ('Instantaneous grid load not covered by renewable generation sources recorded at the end of the following hour.', 'MW'),
    'renewable_share_of_load':          ('Instantaneous grid load covered by renewable generation sources recorded at the end of the following hour.', 'MW'),
    'renewable_share_of_generation':    ('Total electricity generation from renewable generation sources recorded at the end of the following hour.', 'MW'),
    'austria_cbet':                     ('The cross-border electricity trading with Austria recorded at the end of the following hour. Positive values indicate an import of electricity, whereas negative values show electricity exports.', 'GW'),
    'belgium_cbet':                     ('The cross-border electricity trading with Belgium recorded at the end of the following hour. Positive values indicate an import of electricity, whereas negative values show electricity exports.', 'GW'),
    'czech_republic_cbet':              ('The cross-border electricity trading with Czech Republic recorded at the end of the following hour. Positive values indicate an import of electricity, whereas negative values show electricity exports.', 'GW'),
    'denmark_cbet':                     ('The cross-border electricity trading with Denmark recorded at the end of the following hour. Positive values indicate an import of electricity, whereas negative values show electricity exports.', 'GW'),
    'france_cbet':                      ('The cross-border electricity trading with France recorded at the end of the following hour. Positive values indicate an import of electricity, whereas negative values show electricity exports.', 'GW'),
    'luxembourg_cbet':                  ('The cross-border electricity trading with Luxembourg recorded at the end of the following hour. Positive values indicate an import of electricity, whereas negative values show electricity exports.', 'GW'),
    'netherlands_cbet':                 ('The cross-border electricity trading with Netherlands recorded at the end of the following hour. Positive values indicate an import of electricity, whereas negative values show electricity exports.', 'GW'),
    'norway_cbet':                      ('The cross-border electricity trading with Norway recorded at the end of the following hour. Positive values indicate an import of electricity, whereas negative values show electricity exports.', 'GW'),
    'poland_cbet':                      ('The cross-border electricity trading with Poland recorded at the end of the following hour. Positive values indicate an import of electricity, whereas negative values show electricity exports.', 'GW'),
    'sweden_cbet':                      ('The cross-border electricity trading with Sweden recorded at the end of the following hour. Positive values indicate an import of electricity, whereas negative values show electricity exports.', 'GW'),
    'switzerland_cbet':                 ('The cross-border electricity trading with Switzerland recorded at the end of the following hour. Positive values indicate an import of electricity, whereas negative values show electricity exports.', 'GW'),
    'sum_cbet':                         ('The cross-border electricity trading with all other countries recorded at the end of the following hour. Positive values indicate an import of electricity, whereas negative values show electricity exports.', 'GW')
}

data_dict = {}
for var in df.columns:
    data_dict[var] = [df[var].dtype, descriptions_units[var][1], descriptions_units[var][0]]

data_dict_df = pd.DataFrame.from_dict(
    data_dict,
    orient='index',
    columns=['dtype', 'units', 'description']
)
data_dict_df.to_csv('data_dictionary.csv')
print(data_dict_df.shape)
data_dict_df.head(100)